# Production bank: Meta HiDDeN + FNNS repair

This is the previous production notebook with a narrow repair layer for **two methods only**. The frozen source ZIP, all 29 model contracts, messages and PSNR rules are unchanged.

Default: continue the existing **100-image** bank; run only Meta HiDDeN and FNNS. The other 27 methods are not prepared or executed. The final report includes all 29.

Upload `hotfix_hidden_fnns.py` when setup asks (once; cached on Drive afterwards). The original production source ZIP is reused from Drive. Stop the old runner first. Use the same output directory, A100 and previous environment.

Meta: ignore only the generated attenuations .pyc in identity comparison; verify actual source/weights. FNNS: restore exact backup bytes, or archive a damaged image/message pair and rebuild its four modes with the original seed and payload. No retry is triggered by BER. Old evidence stays in `_meta/repairs/`. Local tests do not constitute a completed GPU run.


## 1. Settings

N_IMAGES is the only change needed for the larger cohort. BANK_ROOT must contain the frozen historical _inputs; OUTPUT_ROOT is a separate new bank.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, io

N_IMAGES = 100               # First run: 5. Reviewed large run: 100. Same output directory.
MODELS = ["meta_hidden_revisited", "fnns_omniseal"]             # Repair only these two; do not recompute the other 27.
MAX_HOURS = 16             # Per-session ceiling, not a Colab lifetime guarantee.
DEVICE = "cuda"
CLEANUP = True             # Reclaim owned local model environments between methods.
SOURCE_ARCHIVE = Path("COCO_Embedding_Bank_PRODUCTION_source.zip")
MANUAL_ASSETS_ROOT = ""     # Optional previously downloaded author weights.

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if os.name != "posix":
    raise RuntimeError("Use Colab or Linux/NVIDIA for the full 29-method run. Native Windows is not supported.")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
BANK_ROOT = BASE / "watermarking_embedding_bank_v1"  # Frozen old _inputs only; read-only.
OUTPUT_ROOT = BASE / "watermarking_production_bank_v1"  # New bank; keep for N=100.
SCRATCH_ROOT = Path("/content/wmbench_production_scratch") if IN_COLAB else Path("/workspace/wmbench_scratch")
SETTINGS = dict(bank=str(BANK_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT),
    n_images=N_IMAGES, models=MODELS, device=DEVICE, max_hours=MAX_HOURS, cleanup=CLEANUP,
    download_author_assets=True, manual_assets_root=MANUAL_ASSETS_ROOT)
print("Read-only originals:", BANK_ROOT)
print("Persistent results:", OUTPUT_ROOT)
print(f"{N_IMAGES} images x 5 messages x 29 methods x 4 modes: at most {N_IMAGES*580:,} PNGs.")
print("Each method: save carriers, then blind-decode PNGs. No channel attacks.")


## 2. Restore the unchanged release and the scoped repair helper

Keep the existing OUTPUT_ROOT. No new trial is created and no checksums are disabled. Previously fixed RoSteALS/SepWater setup remains in this notebook unchanged.


In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is not mounted")
if not (BANK_ROOT / "_inputs/manifest.json").is_file():
    raise FileNotFoundError(f"Set BANK_ROOT to the old bank containing _inputs/manifest.json: {BANK_ROOT}")
for left, right in ((BANK_ROOT, OUTPUT_ROOT), (BANK_ROOT, SCRATCH_ROOT), (OUTPUT_ROOT, SCRATCH_ROOT)):
    if left.resolve().is_relative_to(right.resolve()) or right.resolve().is_relative_to(left.resolve()):
        raise ValueError("Old bank, output and scratch must be separate, non-nested directories")
if not shutil.which("git"):
    raise RuntimeError("Git is required for the pinned author sources")
if DEVICE == "cuda" and not shutil.which("nvidia-smi"):
    raise RuntimeError("Select an NVIDIA GPU runtime first")
if (OUTPUT_ROOT / "bank.json").exists():
    raise ValueError("OUTPUT_ROOT points to a legacy bank. Use a NEW folder.")

SOURCE_SHA256 = "6d7ddfcd05abdaf46f97ab7b6b174bff098b3043edcf069db64041554263f7b5"
cached_source = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
source_candidate = SOURCE_ARCHIVE if SOURCE_ARCHIVE.is_file() else cached_source
payload = source_candidate.read_bytes() if source_candidate.is_file() else None
if IN_COLAB and (payload is None or hashlib.sha256(payload).hexdigest() != SOURCE_SHA256):
    from google.colab import files
    print("Upload COCO_Embedding_Bank_PRODUCTION_source.zip supplied with this notebook.")
    uploads = files.upload()
    candidates = [data for data in uploads.values() if hashlib.sha256(data).hexdigest() == SOURCE_SHA256]
    if not candidates:
        raise ValueError("Wrong source ZIP: checksum does not match this notebook")
    payload = candidates[0]
if payload is None or hashlib.sha256(payload).hexdigest() != SOURCE_SHA256:
    raise ValueError("Place the matching ZIP beside the notebook or set SOURCE_ARCHIVE")

cached_source.parent.mkdir(parents=True, exist_ok=True)
if not cached_source.exists():
    cached_source.write_bytes(payload)
# Old author dependencies contain a small native BCH extension.
if IN_COLAB and not shutil.which("g++"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "build-essential"], check=True)
if not shutil.which("g++"):
    raise RuntimeError("Install build-essential on the Linux server before running author environments.")
# Only RoSteALS/SepWater need these author-import system libraries.
# Restore the setup used by RoSteALS_AUTHOR_FIX and RoSteALS_SepWater_RECOVERY.
# No model source, checkpoint, Python lock, payload or matching rule is changed.
import ctypes
import ctypes.util

def ensure_author_system_dependencies(models, in_colab):
    affected = {"rosteals", "sepwater"}
    selected = affected if models == "all" else affected.intersection(models)
    if not selected:
        return {}

    requirements = {
        "libmagickwand-dev": (
            ("MagickWand-6.Q16", "MagickWand-6.Q16HDRI", "MagickWand-7.Q16HDRI",
             "MagickWand-7.Q16", "MagickWand"), "MagickWandGenesis"),
        "libgl1": (("GL",), None),
    }

    def inspect_libraries():
        found = {}
        for package, (names, symbol) in requirements.items():
            for name in names:
                path = ctypes.util.find_library(name)
                if not path:
                    continue
                try:
                    library = ctypes.CDLL(path)
                    if symbol:
                        getattr(library, symbol)
                except (OSError, AttributeError):
                    continue
                found[package] = path
                break
        return found

    found = inspect_libraries()
    missing = [package for package in requirements if package not in found]
    if missing:
        install = ["apt-get", "-qq", "install", "-y", "--no-install-recommends", *missing]
        if not in_colab or not shutil.which("apt-get") or os.geteuid() != 0:
            raise RuntimeError(
                "RoSteALS/SepWater system libraries are missing. On Debian/Ubuntu run: "
                "sudo apt-get update && sudo " + " ".join(install)
                + ". Then rerun this setup cell; keep the existing output directory.")
        print("Installing missing system libraries for " + ", ".join(sorted(selected))
              + ": " + ", ".join(missing), flush=True)
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=600)
        subprocess.run(install, check=True, timeout=600)
        found = inspect_libraries()
        missing = [package for package in requirements if package not in found]
        if missing:
            raise RuntimeError("System-library preflight failed after installation: "
                               + ", ".join(missing) + ". No model has been started.")
    print("RoSteALS/SepWater system-library preflight passed:", found, flush=True)
    return found

AUTHOR_SYSTEM_LIBRARIES = ensure_author_system_dependencies(MODELS, IN_COLAB)

PROJECT = SCRATCH_ROOT / "release" / SOURCE_SHA256[:16]
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        target = (PROJECT / item.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or item.is_dir():
            raise ValueError("Unexpected archive member")
        content = archive.read(item)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Release source modified: {target}; use fresh scratch")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

# Only small controller tools enter this kernel. Each model gets a separate venv.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10", str(controller_dir)], check=True)
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER), "numpy==1.26.4",
                "Pillow==10.4.0", "psutil==6.1.1", "gdown==5.2.0"], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(PROJECT / "src"), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4",
           OPENBLAS_NUM_THREADS="1", PATH=str(Path(UV).parent) + os.pathsep + ENV.get("PATH", ""))
CONFIG = SCRATCH_ROOT / "production_settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Free local SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Ready. Stop any previous runner for this output directory before starting.")

subprocess.run([str(CONTROLLER), "-m", "wmbench_bank.runner", "validate",
                "--settings", str(CONFIG)], env=ENV, check=True)

# Small readable Python repair helper; deliberately outside the frozen release.
HOTFIX_SHA256 = "0d7406e6193f5496fbc60f46157f86130d338eb2bdee8d9d76a7b7a65a5bc85b"
HOTFIX_NAME = "hotfix_hidden_fnns.py"
cached_fix = OUTPUT_ROOT / "_release" / ("hotfix_" + HOTFIX_SHA256[:16]) / HOTFIX_NAME
fix_source = Path(HOTFIX_NAME)
if not fix_source.is_file():
    fix_source = cached_fix
if not fix_source.is_file() and IN_COLAB:
    from google.colab import files
    print("Upload hotfix_hidden_fnns.py from the supplied repair bundle.")
    files.upload()
    fix_source = Path(HOTFIX_NAME)
if not fix_source.is_file():
    raise FileNotFoundError("Place hotfix_hidden_fnns.py beside this notebook and rerun setup.")
fix_bytes = fix_source.read_bytes()
if hashlib.sha256(fix_bytes).hexdigest() != HOTFIX_SHA256:
    raise RuntimeError("Repair helper checksum differs; use the file supplied with this notebook.")
cached_fix.parent.mkdir(parents=True, exist_ok=True)
if not cached_fix.exists():
    cached_fix.write_bytes(fix_bytes)
if hashlib.sha256(cached_fix.read_bytes()).hexdigest() != HOTFIX_SHA256:
    raise RuntimeError("Repair helper did not persist correctly on Drive.")
HOTFIX = SCRATCH_ROOT / ("hotfix_" + HOTFIX_SHA256[:16]) / HOTFIX_NAME
HOTFIX.parent.mkdir(parents=True, exist_ok=True)
HOTFIX.write_bytes(fix_bytes)
print("Scoped repair ready:", MODELS, "Existing results:", OUTPUT_ROOT)


## 3. Create carriers and decode them without attacks

One isolated method at a time. Saved PNGs, keys, messages and clean scores persist on Drive. Nonzero BER is a measurement, not a retry request.

In [ ]:
# After reconnecting: run cells 1-3 again with the SAME paths and source ZIP.
# Finished measurements (including nonzero BER) are reused, not improved by retrying.
command = [str(CONTROLLER), str(HOTFIX), "run", "--settings", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding="utf-8", errors="replace", start_new_session=(os.name != "nt"))
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f"Controller exit={code}; completed results remain in {OUTPUT_ROOT}")
except BaseException:
    if process.poll() is None:
        if os.name != "nt":
            import signal
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
    raise
print("Queue finished or time budget reached. Inspect the report, not only this message.")


## 4. Report

Safe after a partial run. No model is loaded by the reporting step. Complete, error-free decoding and available matched targets are different properties.

In [ ]:
# Read-only aggregation of all 29 methods, including the untouched 27.
REPORT_CONFIG = SCRATCH_ROOT / "production_report_all_settings.json"
REPORT_CONFIG.write_text(json.dumps({**SETTINGS, "models": "all"}, indent=2), encoding="utf-8")
subprocess.run([str(CONTROLLER), "-m", "wmbench_bank.runner", "report",
                "--settings", str(REPORT_CONFIG)], env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "_reports/REPORT.md").read_text(encoding="utf-8")))
summary = json.loads((OUTPUT_ROOT / "_reports/summary.json").read_text())
print("Technically complete:", summary["all_selected_technical_complete"])
print("All matched targets available:", summary["all_targets_available"])
print("Saved PNGs:", summary["saved_carriers"], "/ planned", summary["planned_carriers"])
# Small review archive: messages, records, audit files and logs, NOT model weights or PNG bank.
review = OUTPUT_ROOT / "production_bank_report.zip"
with zipfile.ZipFile(review, "w", zipfile.ZIP_DEFLATED) as archive:
    candidates = [OUTPUT_ROOT / "trial.json"]
    candidates += list((OUTPUT_ROOT / "_reports").rglob("*"))
    for method in json.loads((OUTPUT_ROOT / "trial.json").read_text())["models"]:
        candidates += list((OUTPUT_ROOT / method / "_meta").rglob("*"))
    candidates += list((OUTPUT_ROOT / "_author/_meta").rglob("*"))
    for path in candidates:
        if path.is_file() and path.suffix in {".json", ".csv", ".md", ".txt", ".log", ".lock"}:
            archive.write(path, path.relative_to(OUTPUT_ROOT))
print("Review archive (contains expected/decoded messages):", review)
print("Full bank: copy the ENTIRE output folder to the external SSD, e.g. E:/watermarking_production_bank_v1.")
print("Only after reviewing the canary, set N_IMAGES=100 and rerun the same notebook with the same paths/ZIP.")
